# Wind vs cycling speed calibration

Goal: find how much each m/s of headwind actually slows down riding speed, using real Garmin data matched against historical Open-Meteo wind.

**Method**
1. Parse GPS + speed from a FIT file
2. Group into 15-second segments (steady-state filter: drop stopped or strongly accelerating)
3. Compute travel bearing per segment → headwind component from historical wind
4. Plot measured speed vs headwind component
5. Fit a line → the slope is the empirical `SPEED_HEADWIND_FACTOR`

Run with multiple FIT files from different days/wind conditions to get a more robust estimate.

In [ ]:
import math
import asyncio
from datetime import datetime, timedelta, timezone
from pathlib import Path

import httpx
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from fitparse import FitFile

# ── Resolve project root regardless of where the kernel was started ───────
try:
    # VS Code sets this to the notebook's absolute path
    PROJECT_ROOT = Path(__vsc_ipynb_file__).parent.parent
except NameError:
    # Regular Jupyter: kernel CWD is usually the project root
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / 'data'
FIT_FILES = sorted(DATA_DIR.glob('*.fit'))

print(f"Project root: {PROJECT_ROOT}")
print(f"Found {len(FIT_FILES)} FIT files:")
for f in FIT_FILES:
    print(f"  {f.name}")

# ── Config ────────────────────────────────────────────────────────────────
SEGMENT_SECONDS         = 15     # aggregate into this many seconds per segment
MIN_SPEED_KMH           = 8      # drop segments where you were barely moving
MAX_ACCEL_MS2           = 0.5    # drop segments with strong acceleration (not steady state)
OPEN_METEO_ARCHIVE_URL  = 'https://archive-api.open-meteo.com/v1/archive'
OPEN_METEO_FORECAST_URL = 'https://api.open-meteo.com/v1/forecast'
ARCHIVE_LAG_DAYS        = 5

def pick_api_url(date_str: str) -> str:
    """Return the right Open-Meteo endpoint for a given date string (YYYY-MM-DD)."""
    ride_date = datetime.strptime(date_str, '%Y-%m-%d').date()
    cutoff    = (datetime.now() - timedelta(days=ARCHIVE_LAG_DAYS)).date()
    return OPEN_METEO_ARCHIVE_URL if ride_date <= cutoff else OPEN_METEO_FORECAST_URL

## 1. Parse FIT file → raw point-by-point DataFrame

In [ ]:
SEMICIRCLE = 2**31 / 180  # FIT stores lat/lon in semicircles

def parse_fit(path) -> pd.DataFrame:
    """Extract one row per record message: timestamp, lat, lon, speed_ms, altitude."""
    fit  = FitFile(str(path))
    rows = []
    for msg in fit.get_messages('record'):
        d = {f.name: f.value for f in msg.fields}
        lat = d.get('position_lat')
        lon = d.get('position_long')
        spd = d.get('enhanced_speed')
        ts  = d.get('timestamp')
        if None in (lat, lon, spd, ts):
            continue
        rows.append({
            'timestamp': ts,
            'lat':       lat / SEMICIRCLE,
            'lon':       lon / SEMICIRCLE,
            'speed_ms':  spd,
            'altitude':  d.get('enhanced_altitude'),
        })
    df = pd.DataFrame(rows).sort_values('timestamp').reset_index(drop=True)
    df['speed_kmh'] = df['speed_ms'] * 3.6
    return df

# Parse all files and tag with the filename
frames = []
for path in FIT_FILES:
    df = parse_fit(path)
    df['file'] = Path(path).stem
    frames.append(df)
    print(f"{Path(path).stem}: {len(df)} points, "
          f"{df['timestamp'].iloc[0]} → {df['timestamp'].iloc[-1]}, "
          f"avg {df['speed_kmh'].mean():.1f} km/h")

raw = pd.concat(frames, ignore_index=True)
raw.head()

## 2. Aggregate into 15-second segments + filter noise

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2) -> float:
    R = 6371.0
    φ1, φ2 = math.radians(lat1), math.radians(lat2)
    dφ = math.radians(lat2 - lat1)
    dλ = math.radians(lon2 - lon1)
    a = math.sin(dφ/2)**2 + math.cos(φ1)*math.cos(φ2)*math.sin(dλ/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

def bearing_deg(lat1, lon1, lat2, lon2) -> float:
    dλ = math.radians(lon2 - lon1)
    φ1, φ2 = math.radians(lat1), math.radians(lat2)
    x = math.sin(dλ) * math.cos(φ2)
    y = math.cos(φ1)*math.sin(φ2) - math.sin(φ1)*math.cos(φ2)*math.cos(dλ)
    return (math.degrees(math.atan2(x, y)) + 360) % 360

def make_segments(df: pd.DataFrame, window_s: int = SEGMENT_SECONDS) -> pd.DataFrame:
    """Group consecutive points into fixed-width time windows."""
    df = df.copy()
    df['elapsed'] = (df['timestamp'] - df['timestamp'].iloc[0]).dt.total_seconds()
    df['bucket']  = (df['elapsed'] // window_s).astype(int)

    segs = []
    for (file, bucket), grp in df.groupby(['file', 'bucket']):
        if len(grp) < 2:
            continue
        t_start = grp['timestamp'].iloc[0]
        t_end   = grp['timestamp'].iloc[-1]
        duration = (t_end - t_start).total_seconds()
        if duration < 1:
            continue

        # Use first/last position for bearing; mean lat/lon for wind lookup
        lat1, lon1 = grp['lat'].iloc[0],  grp['lon'].iloc[0]
        lat2, lon2 = grp['lat'].iloc[-1], grp['lon'].iloc[-1]

        avg_speed_ms  = grp['speed_ms'].mean()
        speed_change  = abs(grp['speed_ms'].iloc[-1] - grp['speed_ms'].iloc[0])
        accel         = speed_change / max(duration, 1)

        segs.append({
            'file':         file,
            'timestamp':    t_start,
            'lat':          grp['lat'].mean(),
            'lon':          grp['lon'].mean(),
            'bearing':      bearing_deg(lat1, lon1, lat2, lon2),
            'speed_ms':     avg_speed_ms,
            'speed_kmh':    avg_speed_ms * 3.6,
            'accel_ms2':    accel,
            'duration_s':   duration,
        })

    segs = pd.DataFrame(segs)

    # Filters: moving fast enough, not accelerating hard
    before = len(segs)
    segs = segs[segs['speed_kmh']  >= MIN_SPEED_KMH]
    segs = segs[segs['accel_ms2']  <= MAX_ACCEL_MS2]
    print(f"Segments: {before} → {len(segs)} after filtering "
          f"(dropped {before - len(segs)} stopped/accelerating)")
    return segs.reset_index(drop=True)

segments = make_segments(raw)
segments.head()

## 3. Fetch historical wind from Open-Meteo

We deduplicate by (date, hour, grid cell) so we only make one API call per unique location-hour — a 42 km ride with ~170 segments might only need 10–20 unique API calls.

In [ ]:
async def fetch_wind_for_date(lat: float, lon: float, date: str) -> dict:
    """Fetch hourly wind for one location on one calendar day.
    Automatically uses the archive API for older rides and the forecast
    API for rides from the last ~5 days."""
    url = pick_api_url(date)
    async with httpx.AsyncClient() as client:
        r = await client.get(url, params={
            'latitude':        round(lat, 2),
            'longitude':       round(lon, 2),
            'hourly':          'windspeed_10m,winddirection_10m',
            'wind_speed_unit': 'ms',
            'start_date':      date,
            'end_date':        date,
            'timezone':        'auto',
        }, timeout=15.0)
        r.raise_for_status()
        return r.json()

def interpolate_wind(data: dict, ts: datetime) -> tuple[float, float]:
    """Linear interpolation between the two hourly forecasts bracketing ts.
    Returns (speed_ms, direction_deg)."""
    times  = data['hourly']['time']
    speeds = data['hourly']['windspeed_10m']
    dirs   = data['hourly']['winddirection_10m']

    floor_str = ts.strftime('%Y-%m-%dT%H:00')
    idx0 = times.index(floor_str) if floor_str in times else 0
    idx1 = min(idx0 + 1, len(times) - 1)
    alpha = (ts.minute + ts.second / 60) / 60

    # Interpolate in (u,v) space for correct circular handling
    u0 = speeds[idx0] * math.sin(math.radians(dirs[idx0]))
    v0 = speeds[idx0] * math.cos(math.radians(dirs[idx0]))
    u1 = speeds[idx1] * math.sin(math.radians(dirs[idx1]))
    v1 = speeds[idx1] * math.cos(math.radians(dirs[idx1]))
    u  = u0 + alpha * (u1 - u0)
    v  = v0 + alpha * (v1 - v0)

    speed = math.sqrt(u**2 + v**2)
    direc = (math.degrees(math.atan2(u, v)) + 360) % 360
    return speed, direc

async def fetch_all_wind(segs: pd.DataFrame) -> dict:
    """Fetch wind for all unique (rounded lat, rounded lon, date) combos in parallel."""
    keys = set()
    for _, row in segs.iterrows():
        key = (round(row['lat'], 2), round(row['lon'], 2),
               row['timestamp'].strftime('%Y-%m-%d'))
        keys.add(key)

    # Show which API each date will use
    dates = {k[2] for k in keys}
    for d in sorted(dates):
        api = 'forecast' if pick_api_url(d) == OPEN_METEO_FORECAST_URL else 'archive'
        print(f"  {d} → {api} API")

    print(f"\nFetching wind for {len(keys)} unique location-days…")
    cache = {}
    sem   = asyncio.Semaphore(5)

    async def bounded(key):
        async with sem:
            cache[key] = await fetch_wind_for_date(key[0], key[1], key[2])

    await asyncio.gather(*[bounded(k) for k in keys])
    print(f"Done — {len(cache)} responses cached.")
    return cache

wind_cache = await fetch_all_wind(segments)

## 4. Compute headwind component per segment

In [ ]:
def headwind_ms(wind_speed: float, wind_dir: float, travel_bearing: float) -> float:
    """
    Project the wind vector onto the travel direction.
    Positive = headwind (slows you down), negative = tailwind (speeds you up).
    """
    dir_rad = math.radians(wind_dir)
    t_rad   = math.radians(travel_bearing)
    u = wind_speed * math.sin(dir_rad)
    v = wind_speed * math.cos(dir_rad)
    return u * math.sin(t_rad) + v * math.cos(t_rad)

hw_list = []
ws_list = []
wd_list = []

for _, row in segments.iterrows():
    key  = (round(row['lat'], 2), round(row['lon'], 2),
            row['timestamp'].strftime('%Y-%m-%d'))
    data = wind_cache.get(key)
    if data is None:
        hw_list.append(None); ws_list.append(None); wd_list.append(None)
        continue
    spd, direc = interpolate_wind(data, row['timestamp'])
    hw_list.append(headwind_ms(spd, direc, row['bearing']))
    ws_list.append(spd)
    wd_list.append(direc)

segments = segments.copy()
segments['wind_speed_ms'] = ws_list
segments['wind_dir_deg']  = wd_list
segments['headwind_ms']   = hw_list

segments = segments.dropna(subset=['headwind_ms']).reset_index(drop=True)
print(f"Segments with wind data: {len(segments)}")
segments[['speed_kmh', 'wind_speed_ms', 'headwind_ms']].describe().round(2)

## 5. Explore: speed vs headwind component

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: scatter with linear fit ─────────────────────────────────────────
ax = axes[0]
ax.scatter(segments['headwind_ms'], segments['speed_kmh'],
           alpha=0.3, s=20, color='steelblue', label='15-s segment')

import numpy as np
x = segments['headwind_ms'].values
y = segments['speed_kmh'].values
coeffs = np.polyfit(x, y, 1)
slope, intercept = coeffs
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line, np.polyval(coeffs, x_line), color='tomato', lw=2,
        label=f'Linear fit  slope={slope:.2f} km/h per m/s')

factor_kmh = slope
factor_ms  = slope / 3.6
print(f"Empirical factor: {factor_kmh:.2f} km/h per m/s headwind")
print(f"  → SPEED_HEADWIND_FACTOR = {abs(factor_ms):.3f}  (current: 0.500)")
print(f"  → Neutral speed from intercept: {intercept:.1f} km/h")

ax.axvline(0, color='gray', lw=0.8, ls='--')
ax.set_xlabel('Headwind component (m/s)\n← tailwind    headwind →')
ax.set_ylabel('Measured speed (km/h)')
ax.set_title('Measured speed vs headwind')
ax.legend()
ax.grid(True, alpha=0.3)

# ── Right: binned means ± std ─────────────────────────────────────────────
ax = axes[1]
bins = np.arange(x.min() - 0.5, x.max() + 1.5, 1.0)
labels = (bins[:-1] + bins[1:]) / 2
seg_copy = segments.copy()
seg_copy['hw_bin'] = pd.cut(seg_copy['headwind_ms'], bins=bins, labels=labels)
binned = seg_copy.groupby('hw_bin')['speed_kmh'].agg(['mean', 'std', 'count']).dropna()
binned = binned[binned['count'] >= 3]

ax.errorbar(binned.index.astype(float), binned['mean'], yerr=binned['std'],
            fmt='o-', color='steelblue', capsize=4, label='Mean ± 1 std')
ax.plot(x_line, np.polyval(coeffs, x_line), color='tomato', lw=1.5,
        ls='--', label='Linear fit')
ax.axvline(0, color='gray', lw=0.8, ls='--')
ax.set_xlabel('Headwind component (m/s)\n← tailwind    headwind →')
ax.set_ylabel('Measured speed (km/h)')
ax.set_title('Binned means (1 m/s bins, n ≥ 3)')
ax.legend()
ax.grid(True, alpha=0.3)

for hw, row in binned.iterrows():
    ax.annotate(f"n={row['count']:.0f}", xy=(float(hw), row['mean']),
                xytext=(0, 8), textcoords='offset points',
                ha='center', fontsize=7, color='gray')

plt.tight_layout()
out_path = PROJECT_ROOT / 'notebooks' / 'wind_speed_scatter.png'
plt.savefig(out_path, dpi=150)
plt.show()
print(f"Plot saved to {out_path}")

## 6. Check symmetry: is headwind worse than tailwind is good?

In [ ]:
# Split into headwind / crosswind / tailwind thirds
import numpy as np

s = segments.copy()
s['type'] = pd.cut(s['headwind_ms'],
                   bins=[-99, -1, 1, 99],
                   labels=['Tailwind (< −1 m/s)', 'Crosswind (±1 m/s)', 'Headwind (> +1 m/s)'])

summary = s.groupby('type')['speed_kmh'].agg(['mean', 'median', 'std', 'count'])
print(summary.round(2))
print()

neutral = s[s['type'] == 'Crosswind (±1 m/s)']['speed_kmh'].mean()
tail    = s[s['type'] == 'Tailwind (< −1 m/s)']['speed_kmh'].mean()
head    = s[s['type'] == 'Headwind (> +1 m/s)']['speed_kmh'].mean()
print(f"Neutral (crosswind) avg speed: {neutral:.1f} km/h")
print(f"Tailwind boost:  +{tail - neutral:.1f} km/h above neutral")
print(f"Headwind penalty: {head - neutral:.1f} km/h below neutral")
if abs(head - neutral) > abs(tail - neutral):
    print("→ Headwind penalty is LARGER than tailwind boost (asymmetric)")
else:
    print("→ Tailwind boost is LARGER than headwind penalty (asymmetric)")

## 7. Summary + recommended factor

In [ ]:
import numpy as np

print("=" * 55)
print("SUMMARY")
print("=" * 55)
print(f"Activities analysed:     {segments['file'].nunique()}")
print(f"Segments (15 s):         {len(segments)}")
print(f"Wind speed range:        {segments['wind_speed_ms'].min():.1f} – "
      f"{segments['wind_speed_ms'].max():.1f} m/s")
print(f"Headwind range:          {segments['headwind_ms'].min():.1f} – "
      f"{segments['headwind_ms'].max():.1f} m/s")
print()
print(f"Linear fit:")
print(f"  intercept (neutral speed): {intercept:.1f} km/h")
print(f"  slope:                     {slope:.2f} km/h per m/s headwind")
print(f"  → SPEED_HEADWIND_FACTOR:   {abs(factor_ms):.3f}  (current value: 0.500)")
print()
print("To apply: update SPEED_HEADWIND_FACTOR in src/config.py")
print("=" * 55)